In [ ]:
# SPR 2026 – BERTimbau MAX_LEN=512 + Per-Class Temperature Scaling
# Estratégia: Aplicar temperaturas diferentes por classe.
# Classes minoritárias (5, 6) recebem T=0.7 (aguça confiança → menos conservador),
# classes majoritárias (0-4) mantêm T=0.958 do winner.
# Hipótese: reduzir temperatura nas classes raras ajuda a não perdê-las.
# Todo o resto idêntico ao winner (0.84027): model-v4, 5 folds, thresholds.

import os
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════════
MAX_LEN     = 512
NUM_CLASSES = 7
N_FOLDS     = 5
BATCH_SIZE  = 16
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

WEIGHTS_BASE = Path('/kaggle/input/datasets/gabrielsavio/model-v4/advanced_outputs_kaggle_4')
CONFIG_KEY   = 'bertimbau_large__cb_focal'
weights_dir  = WEIGHTS_BASE / 'weights' / CONFIG_KEY

# Thresholds do winner (0.84027) — mantidos
BEST_THRESHOLDS = [0.9500, 1.6000, 1.3500, 1.0, 0.4000, 1.1500, 0.1]

# Per-class temperatures:
# Classes 0-4 (majoritárias): T=0.958 (temperatura original do winner)
# Classes 5, 6 (raras: BI-RADS Suspeita/Maligno): T=0.7 (mais agressivo)
PER_CLASS_TEMPS = [0.958, 0.958, 0.958, 0.958, 0.958, 0.7, 0.7]

print(f'Device: {DEVICE}')
print(f'Per-class temperatures: {PER_CLASS_TEMPS}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET & PRÉ-PROCESSAMENTO
# ═══════════════════════════════════════════════════════════════════════════════
class MammographyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        return item


INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CALIBRATION FUNCTIONS — VERSÃO PER-CLASS
# ═══════════════════════════════════════════════════════════════════════════════
def per_class_temperature_scale(probs, temps_per_class):
    """Aplica temperatura diferente para cada classe.
    Opera no espaço de log-probabilidades (logits) por coluna.
    """
    logits = np.log(probs + 1e-10)             # (N, 7)
    temps  = np.array(temps_per_class)         # (7,)
    scaled = logits / temps[np.newaxis, :]     # broadcast: (N, 7) / (1, 7)
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        outputs = model(**kwargs)
        probs   = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)


# ═══════════════════════════════════════════════════════════════════════════════
# CARREGAR DADOS DE TESTE
# ═══════════════════════════════════════════════════════════════════════════════
test_path  = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv')
test_df    = pd.read_csv(test_path)
test_texts = [build_input_text(t) for t in test_df['report'].tolist()]
print(f'Test samples: {len(test_df)}')

tokenizer = AutoTokenizer.from_pretrained(weights_dir / 'tokenizer')
dataset   = MammographyDataset(test_texts, tokenizer, MAX_LEN)
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=0, pin_memory=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 5-FOLD ENSEMBLE INFERENCE (model-v4) — idêntico ao winner
# ═══════════════════════════════════════════════════════════════════════════════
config_path  = weights_dir / 'model_config'
test_probs   = np.zeros((len(test_df), NUM_CLASSES))
folds_loaded = 0

for fold in range(N_FOLDS):
    weight_path = weights_dir / f'fold_{fold}.pt'
    if not weight_path.exists():
        print(f'Fold {fold} não encontrado, pulando...')
        continue
    print(f'Carregando fold {fold}...', end=' ')
    config = AutoConfig.from_pretrained(config_path, num_labels=NUM_CLASSES)
    model  = AutoModelForSequenceClassification.from_config(config).to(DEVICE)
    state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)
    fold_probs   = predict(model, loader)
    test_probs  += fold_probs
    folds_loaded += 1
    print(f'ok')
    del model, state_dict
    torch.cuda.empty_cache()

test_probs /= folds_loaded
print(f'{folds_loaded} folds carregados.')

# ═══════════════════════════════════════════════════════════════════════════════
# PER-CLASS TEMPERATURE SCALING → THRESHOLDS → SUBMISSÃO
# ═══════════════════════════════════════════════════════════════════════════════
calibrated_probs = per_class_temperature_scale(test_probs, PER_CLASS_TEMPS)
predictions      = apply_thresholds(calibrated_probs, BEST_THRESHOLDS)

print(f'\nProbabilidades calibradas (amostra):')
print(np.round(calibrated_probs, 4))

submission = pd.DataFrame({'ID': test_df['ID'], 'target': predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('\n=== SUBMISSION ===')
print(submission.to_string(index=False))
print(f'submission.csv salvo ({len(submission)} linhas)')